# 📝 Automatic Grammar Error Detector
### NLP Course Project — a rule-based checker built from fundamentals with spaCy

**Goal:** detect common English grammar mistakes, explain *why* each one is wrong, suggest a fix, and measure how well the detector works.

**Approach.** We do **not** use a pretrained grammar-correction model or an off-the-shelf checker (LanguageTool, GPT, etc.). spaCy only gives us *linguistic annotations* — tokens, lemmas, part-of-speech (POS) tags, morphology and a dependency parse. **Every grammar rule is written by us** as a small, readable function, so each decision can be explained.

```
 raw text
    │
    ▼
 ┌──────────────────────────────────────────┐
 │ spaCy pipeline                           │   ← linguistic features
 │ tokenizer → tagger → parser → lemmatizer │
 └──────────────────────────────────────────┘
    │  Doc (tokens + POS + dependencies)
    ▼
 ┌──────────────────────────────────────────┐
 │ 10 rule functions (written by us)        │   ← the "grammar knowledge"
 │ SVA · ART · REP · MOD · TO · NUM ·       │
 │ CONF · LEX · CAP · PUN                   │
 └──────────────────────────────────────────┘
    │  list of GrammarError objects
    ▼
 report  ·  highlighted HTML  ·  auto-correction  ·  evaluation (P / R / F1)
```

**Notebook outline**
1. Setup
2. spaCy primer — what features do we get?
3. Building blocks — the `GrammarError` class and word-form helpers
4. The 10 grammar rules (each: idea → code → demo)
5. Putting it together — `GrammarChecker`
6. Try it yourself
7. Evaluation — precision, recall, F1
8. Error analysis & limitations
9. Future work

## 1. Setup

In [ ]:
# Run once (Colab / fresh environment). Safe to re-run.
!pip install -q spacy pandas
!python -m spacy download en_core_web_sm -q

In [2]:
import re
import html
from dataclasses import dataclass
from typing import Optional, List

import spacy
import pandas as pd
from IPython.display import HTML, display

nlp = spacy.load("en_core_web_sm")      # small English model: tagger + parser + lemmatizer
print("spaCy version:", spacy.__version__)
print("Pipeline     :", nlp.pipe_names)

spaCy version: 3.8.16
Pipeline     : ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 2. spaCy primer — what features do we get?

Before writing rules, let's look at what spaCy tells us about a sentence. Notice the **wrong** sentence below: `She go` — the verb should be `goes`.

In [3]:
def show_tokens(text):
    doc = nlp(text)
    print(f"{'TOKEN':<12}{'LEMMA':<12}{'POS':<8}{'TAG':<6}{'DEP':<10}{'HEAD':<10}")
    print("-" * 58)
    for t in doc:
        print(f"{t.text:<12}{t.lemma_:<12}{t.pos_:<8}{t.tag_:<6}{t.dep_:<10}{t.head.text:<10}")

show_tokens("She go to schools every day.")

TOKEN       LEMMA       POS     TAG   DEP       HEAD      
----------------------------------------------------------
She         she         PRON    PRP   nsubj     go        
go          go          VERB    VBP   ROOT      go        
to          to          ADP     IN    prep      go        
schools     school      NOUN    NNS   pobj      to        
every       every       DET     DT    det       day       
day         day         NOUN    NN    npadvmod  go        
.           .           PUNCT   .     punct     go        


**How to read this table**

| Field | Meaning | Used for |
|---|---|---|
| `lemma_` | dictionary form (`goes → go`) | building corrections |
| `pos_` | coarse part of speech (`NOUN`, `VERB`, …) | finding verbs / nouns |
| `tag_` | **fine-grained Penn Treebank tag** | detecting *form* errors |
| `dep_` | grammatical role (`nsubj` = subject, `det` = determiner …) | linking subject ↔ verb, determiner ↔ noun |
| `head` | the word this token depends on | "which verb belongs to this subject?" |

**The Penn Treebank tags we rely on**

| Tag | Meaning | Example |
|---|---|---|
| `NN` / `NNS` | singular / plural common noun | *book* / *books* |
| `VB` | base verb | *go* |
| `VBZ` | 3rd-person singular present | *goes* |
| `VBP` | non-3rd-person present | *go* (with I/you/we/they) |
| `VBD` | past tense | *went* |
| `VBG` / `VBN` | -ing form / past participle | *going* / *gone* |
| `MD` | modal verb | *can, will, must* |
| `TO` | infinitive marker | *to* (in *to go*) |

> 💡 **Key insight:** the tagger labels `go` after `She` as **VBP** (a *plural-type* present verb), while the parser says `She` is the **subject (nsubj)** of that verb. A mismatch between *subject number* and *verb tag* = agreement error. That is our first rule.

## 3. Building blocks

### 3.1 The `GrammarError` object
Every rule returns a list of `GrammarError`s. Using one common structure lets us reuse the same code for **reporting, highlighting and correcting**.

In [4]:
@dataclass
class GrammarError:
    rule: str                    # short code, e.g. "SVA"
    message: str                 # human-readable explanation
    start: int                   # character offset where the error starts
    end: int                     # character offset where it ends
    original: str                # the offending text
    suggestion: Optional[str] = None   # proposed replacement (None = no automatic fix)


def make_error(rule, message, first, last=None, suggestion=None):
    """Build a GrammarError that covers spaCy tokens `first` … `last`."""
    if last is None:
        last = first
    start = first.idx
    end = last.idx + len(last.text)
    return GrammarError(rule, message, start, end, first.doc.text[start:end], suggestion)

### 3.2 Word-form helpers
To *suggest* a correction we sometimes need to build a new word form (`go → goes`, `book → books`). These tiny helpers use simple English spelling rules.

In [5]:
def match_case(original: str, new: str) -> str:
    """Copy the capitalisation of `original` onto `new` (e.g. 'Go' → 'Goes')."""
    if not new:
        return new
    if original.isupper() and len(original) > 1:
        return new.upper()
    if original[0].isupper():
        return new[0].upper() + new[1:]
    return new


def add_s(word: str) -> str:
    """Regular -s / -es / -ies rule (used for 3rd-person verbs and plural nouns)."""
    if re.search(r"(s|x|z|ch|sh)$", word):
        return word + "es"
    if re.search(r"[^aeiou]y$", word):
        return word[:-1] + "ies"
    return word + "s"


def third_person(verb_lemma: str) -> str:
    irregular = {"be": "is", "have": "has", "do": "does", "go": "goes"}
    return irregular.get(verb_lemma, add_s(verb_lemma))


def pluralize(noun: str) -> str:
    irregular = {"child": "children", "man": "men", "woman": "women",
                 "person": "people", "foot": "feet", "tooth": "teeth", "mouse": "mice"}
    return irregular.get(noun.lower(), add_s(noun))


print(third_person("go"), third_person("study"), third_person("watch"), third_person("have"))
print(pluralize("book"), pluralize("city"), pluralize("box"), pluralize("child"))

goes studies watches has
books cities boxes children


### 3.3 A tiny demo helper
So we can test each rule on its own as we build it.

In [6]:
def demo(rule, sentences):
    """Run ONE rule function on a few sentences and print what it finds."""
    for s in sentences:
        errs = rule(nlp(s))
        print(("✗ " if errs else "✓ ") + s)
        for e in errs:
            print(f"     └─ [{e.rule}] '{e.original}' → {e.suggestion!r}   ({e.message})")

## 4. The grammar rules

Each rule is a function `rule(doc) -> list[GrammarError]`. We go from the most powerful (uses the dependency parse) to the simplest (surface patterns).

---
### Rule 1 — Subject–Verb Agreement (`SVA`)

**Grammar fact:** a singular subject takes a singular verb (*He goes*), a plural subject takes a plural verb (*They go*). `I` and `you` have their own forms (*I am*, *you are*).

**Algorithm**
1. Find every token whose dependency is `nsubj` / `nsubjpass` (a subject). For *"There is many books"* the real subject is the noun after the expletive *there*.
2. Decide the subject's **number**: `I`, `you`, singular, or plural (pronoun lists, noun tags `NN`/`NNS`, and `X and Y` ⇒ plural).
3. Find the **finite verb** that belongs to it (the verb itself, or its first auxiliary like *do/is/has*).
4. Compare the verb's form with what that subject requires. If they differ → error.

**Why it is explainable:** the message tells you the subject, its number, and the verb that disagrees.

In [7]:
SINGULAR_PRONOUNS = {"he", "she", "it", "this", "that", "everyone", "everybody", "someone",
                     "somebody", "anyone", "anybody", "nobody", "each", "either", "neither", "one"}
PLURAL_PRONOUNS   = {"we", "they", "these", "those", "both", "many", "several"}
# Nouns where number is unclear — we stay silent instead of risking a false alarm
AMBIGUOUS_SUBJECTS = {"series", "species", "number", "lot", "couple", "bunch", "variety",
                      "majority", "minority", "rest", "group", "team", "family", "staff"}
LABEL = {"1sg": "1st-person singular", "2": "2nd person", "sg": "singular", "pl": "plural"}
FINITE = ("VBZ", "VBP", "VBD")      # tags of verbs that carry tense + agreement


def subject_number(subj):
    """Return '1sg' | '2' | 'sg' | 'pl' | None (None = unsure → don't judge)."""
    low = subj.lower_
    if low == "i":                      return "1sg"
    if low == "you":                    return "2"
    if low in SINGULAR_PRONOUNS:        return "sg"
    if low in PLURAL_PRONOUNS:          return "pl"
    if low in AMBIGUOUS_SUBJECTS:       return None
    if subj.tag_ in ("WDT", "WP", "EX"):return None          # who / which / there
    # "Tom and Jerry" → plural
    conjs = [c for c in subj.children if c.dep_ == "conj"]
    if conjs:
        helpers = list(subj.children) + [x for c in conjs for x in c.children]
        if any(t.dep_ == "cc" and t.lower_ == "and" for t in helpers):
            return "pl"
    # "These book are…": trust the determiner here; the noun error itself is reported by rule NUM
    if any(c.lower_ in PLURAL_PRONOUNS and c.dep_ in ("det", "amod", "nummod", "predet") for c in subj.children):
        return "pl"
    if subj.tag_ in ("NNS", "NNPS"):    return "pl"
    if subj.tag_ in ("NN", "NNP"):      return "sg"
    return None


def finite_verb(subj):
    """The finite verb (or first auxiliary) that must agree with this subject."""
    head = subj.head
    if head.pos_ not in ("VERB", "AUX"):
        return None
    auxes = sorted([c for c in head.children if c.dep_ in ("aux", "auxpass")], key=lambda t: t.i)
    if auxes:
        return auxes[0] if auxes[0].tag_ in FINITE else None   # modal (can/will) → no agreement
    return head if head.tag_ in FINITE else None


def expected_form(verb, number):
    """What should this verb look like for a subject of this number?"""
    lemma, word = verb.lemma_.lower(), verb.lower_
    past = verb.tag_ == "VBD"
    if lemma == "be":
        table = {"1sg": ("am", "was"), "sg": ("is", "was"), "2": ("are", "were"), "pl": ("are", "were")}
        return table[number][1 if past else 0]
    if lemma in ("have", "do"):
        if past:
            return word                                   # had / did: same for everyone
        return third_person(lemma) if number == "sg" else lemma
    if verb.tag_ in ("VBZ", "VBP"):
        return third_person(lemma) if number == "sg" else lemma
    return None                                           # past tense of normal verbs: no agreement


def check_subject_verb(doc):
    errors = []
    for subj in doc:
        # normal subject, OR the "real" subject after expletive 'there' ("There is many books")
        is_subject = subj.dep_ in ("nsubj", "nsubjpass") or \
                     (subj.dep_ == "attr" and any(c.dep_ == "expl" for c in subj.head.children))
        if not is_subject:
            continue
        number, verb = subject_number(subj), finite_verb(subj)
        if number is None or verb is None or verb.text.startswith("'"):
            continue
        # "If I were you…" is correct subjunctive — skip
        before = {t.lower_ for t in subj.sent if t.i < verb.i}
        if verb.lower_ == "were" and before & {"if", "wish", "though"}:
            continue
        fix = expected_form(verb, number)
        if fix and fix != verb.lower_:
            errors.append(make_error(
                "SVA",
                f"Subject '{subj.text}' is {LABEL[number]}, but verb '{verb.text}' does not agree with it",
                verb, suggestion=match_case(verb.text, fix)))
    return errors


demo(check_subject_verb, [
    "He go to school every day.",
    "The students was late for class.",
    "She don't like coffee.",
    "I is ready.",
    "There is many books on the shelf.",
    "My brother and sister likes pizza.",
    "The cat sleeps on the sofa.",          # correct
    "If I were you, I would study harder.", # correct (subjunctive)
])

✗ He go to school every day.
     └─ [SVA] 'go' → 'goes'   (Subject 'He' is singular, but verb 'go' does not agree with it)
✗ The students was late for class.
     └─ [SVA] 'was' → 'were'   (Subject 'students' is plural, but verb 'was' does not agree with it)
✗ She don't like coffee.
     └─ [SVA] 'do' → 'does'   (Subject 'She' is singular, but verb 'do' does not agree with it)
✗ I is ready.
     └─ [SVA] 'is' → 'am'   (Subject 'I' is 1st-person singular, but verb 'is' does not agree with it)
✗ There is many books on the shelf.
     └─ [SVA] 'is' → 'are'   (Subject 'books' is plural, but verb 'is' does not agree with it)
✗ My brother and sister likes pizza.
     └─ [SVA] 'likes' → 'like'   (Subject 'brother' is plural, but verb 'likes' does not agree with it)
✓ The cat sleeps on the sofa.
✓ If I were you, I would study harder.


---
### Rule 2 — Article choice: *a* vs *an* (`ART`)

**Grammar fact:** use **an** before a *vowel sound*, **a** before a *consonant sound*. It is about **sound**, not spelling: *an hour* (silent h), *a university* (starts with "yoo").

**Algorithm:** look at the word right after `a/an`; guess its sound from the first letters plus two small exception lists.

In [8]:
VOWEL_SOUND_PREFIXES      = ("hour", "honest", "honor", "honour", "heir")      # silent h → "an"
CONSONANT_SOUND_PREFIXES  = ("univers", "unicorn", "uniform", "unique", "union", "unit", "use", "usu",
                             "util", "eu", "one", "once")                       # "yoo"/"w" sound → "a"


def starts_with_vowel_sound(word: str) -> bool:
    w = word.lower()
    if w.startswith(VOWEL_SOUND_PREFIXES):
        return True
    if w.startswith(CONSONANT_SOUND_PREFIXES):
        return False
    return w[0] in "aeiou"


def check_articles(doc):
    errors = []
    for tok in doc[:-1]:
        if tok.lower_ not in ("a", "an"):
            continue
        nxt = doc[tok.i + 1]
        if not nxt.is_alpha or nxt.text.isupper():          # skip numbers / acronyms (FBI, MBA…)
            continue
        needed = "an" if starts_with_vowel_sound(nxt.text) else "a"
        if tok.lower_ != needed:
            errors.append(make_error(
                "ART", f"Use '{needed}' before '{nxt.text}' (it starts with a "
                       f"{'vowel' if needed == 'an' else 'consonant'} sound)",
                tok, suggestion=match_case(tok.text, needed)))
    return errors


demo(check_articles, [
    "She bought a apple.",
    "He is an university student.",
    "It took an hour.",             # correct (silent h)
    "A university is big.",         # correct ("yoo" sound)
])

✗ She bought a apple.
     └─ [ART] 'a' → 'an'   (Use 'an' before 'apple' (it starts with a vowel sound))
✗ He is an university student.
     └─ [ART] 'an' → 'a'   (Use 'a' before 'university' (it starts with a consonant sound))
✓ It took an hour.
✓ A university is big.


---
### Rule 3 — Repeated words (`REP`)

**Pattern:** the same word twice in a row (*"to to"*, *"the the"*) — usually a typing slip. A tiny whitelist keeps legal repeats like *"had had"* and *"that that"*.

In [9]:
LEGAL_REPEATS = {"had", "that"}

def check_repeated_words(doc):
    errors = []
    for a, b in zip(doc, doc[1:]):
        if a.is_alpha and a.lower_ == b.lower_ and a.lower_ not in LEGAL_REPEATS:
            errors.append(make_error("REP", f"The word '{a.text}' is repeated",
                                     a, b, suggestion=a.text))
    return errors


demo(check_repeated_words, ["I went to to the market.", "This is the the best project.",
                            "He said that that was fine."])

✗ I went to to the market.
     └─ [REP] 'to to' → 'to'   (The word 'to' is repeated)
✗ This is the the best project.
     └─ [REP] 'the the' → 'the'   (The word 'the' is repeated)
✓ He said that that was fine.


---
### Rule 4 — Verb form after a modal (`MOD`)

**Grammar fact:** after a modal (*can, will, must, should, might…*) the verb must be in its **base form**: *can **go***, not *can goes / can went / can going*.

**Algorithm:** find tokens tagged `MD`; look at the next word (skipping adverbs like *not/never*); if that verb is **not in base form** → error. Fix = the verb's **lemma**.

> ⚠️ **Lesson learned while testing:** spaCy's tagger is *context-aware* — after `will` it happily tags the wrong word `went` as `VB` (base form), and the lemmatizer then keeps `went` as its own lemma! So we cannot trust the tag or the in-context lemma. Instead we look the word up **in isolation** (`isolated_lemma`) and compare **surface form vs lemma**: `went ≠ go` ⇒ inflected ⇒ error, while `go = go` ⇒ fine.

In [10]:
from functools import lru_cache

@lru_cache(maxsize=None)
def isolated_lemma(word: str) -> str:
    """Lemma of a word looked up ALONE (no sentence context), e.g. 'went' → 'go'."""
    return nlp(word)[0].lemma_.lower()


def is_inflected_verb(tok):
    """True if the verb is NOT in base form (surface form differs from its lemma: went≠go, goes≠go)."""
    return tok.pos_ in ("VERB", "AUX") and tok.lower_ != isolated_lemma(tok.lower_)


print(isolated_lemma("went"), isolated_lemma("goes"), isolated_lemma("finished"), isolated_lemma("go"))


def check_modal_verb(doc):
    errors = []
    for tok in doc:
        if tok.tag_ != "MD":
            continue
        j = tok.i + 1
        while j < len(doc) and (doc[j].pos_ == "ADV" or doc[j].lower_ in ("n't", "not")):
            j += 1                                    # skip "can *never* go"
        if j < len(doc) and is_inflected_verb(doc[j]):
            v = doc[j]
            errors.append(make_error(
                "MOD", f"After the modal '{tok.text}' use the base form of the verb, not '{v.text}'",
                v, suggestion=match_case(v.text, isolated_lemma(v.lower_))))
    return errors


demo(check_modal_verb, [
    "She can goes to the party.",
    "He will went home tomorrow.",
    "We must finished the work today.",
    "She will be going to Chennai.",      # correct
    "We should have gone earlier.",       # correct
])

go go finish go
✗ She can goes to the party.
     └─ [MOD] 'goes' → 'go'   (After the modal 'can' use the base form of the verb, not 'goes')
✗ He will went home tomorrow.
     └─ [MOD] 'went' → 'go'   (After the modal 'will' use the base form of the verb, not 'went')
✗ We must finished the work today.
     └─ [MOD] 'finished' → 'finish'   (After the modal 'must' use the base form of the verb, not 'finished')
✓ She will be going to Chennai.
✓ We should have gone earlier.


---
### Rule 5 — Verb form after infinitive *to* (`TO`)

**Grammar fact:** *to* + **base verb** (*want to go*). Never *to went / to goes*.

**Algorithm:** token tagged `TO` followed by a verb that is not in base form (same isolated surface-vs-lemma test as Rule 4). Preposition *to* is tagged `IN`, so *"went to school"* is not touched.

In [11]:
def check_infinitive(doc):
    errors = []
    for tok in doc[:-1]:
        nxt = doc[tok.i + 1]
        if tok.tag_ == "TO" and is_inflected_verb(nxt):
            errors.append(make_error(
                "TO", f"After infinitive 'to' use the base form, not '{nxt.text}'",
                nxt, suggestion=match_case(nxt.text, isolated_lemma(nxt.lower_))))
    return errors


demo(check_infinitive, ["I want to went home.", "She decided to goes there.",
                        "I want to learn NLP.", "He went to school."])

✗ I want to went home.
     └─ [TO] 'went' → 'go'   (After infinitive 'to' use the base form, not 'went')
✗ She decided to goes there.
     └─ [TO] 'goes' → 'go'   (After infinitive 'to' use the base form, not 'goes')


✓ I want to learn NLP.
✓ He went to school.


---
### Rule 6 — Determiner–noun number agreement (`NUM`)

**Grammar fact:** some determiners only go with **singular** nouns (*a, an, each, every, this, that, one, another*) and some only with **plural** nouns (*these, those, many, several, both*).

**Algorithm:** for each such determiner, look at the noun it modifies (`token.head`) and check its tag (`NN` vs `NNS`). Fix = singularise with the **lemma**, or pluralise with `pluralize()`.

**Safety guards:** phrases like *"a few books"*, *"a lot of"* and nouns such as *series/species* look like mismatches but are fine, so we skip them.

In [12]:
SINGULAR_DETS = {"a", "an", "each", "every", "this", "that", "one", "another"}
PLURAL_DETS   = {"these", "those", "many", "several", "both"}
QUANTITY_WORDS = {"few", "couple", "lot", "number", "dozen", "hundred", "thousand", "million", "bit", "little"}
SAME_FORM_NOUNS = {"series", "species", "means", "news", "mathematics", "physics", "politics"}


def check_determiner_noun(doc):
    errors = []
    for det in doc[:-1]:
        low = det.lower_
        if low not in SINGULAR_DETS | PLURAL_DETS:
            continue
        noun = det.head
        if noun.i <= det.i or noun.tag_ not in ("NN", "NNS") or noun.lower_ in SAME_FORM_NOUNS:
            continue
        if det.dep_ not in ("det", "amod", "nummod", "predet"):
            continue
        if doc[det.i + 1].lower_ in QUANTITY_WORDS:             # "a few books", "a lot"
            continue

        if low in SINGULAR_DETS and noun.tag_ == "NNS":
            errors.append(make_error(
                "NUM", f"'{det.text}' needs a singular noun, but '{noun.text}' is plural",
                noun, suggestion=match_case(noun.text, noun.lemma_)))
        elif low in PLURAL_DETS and noun.tag_ == "NN":
            errors.append(make_error(
                "NUM", f"'{det.text}' needs a plural noun, but '{noun.text}' is singular",
                noun, suggestion=match_case(noun.text, pluralize(noun.text))))
    return errors


demo(check_determiner_noun, [
    "I read a books yesterday.",
    "These book are interesting.",
    "Every students must attend.",
    "This apples are fresh.",
    "I read a few books.",        # correct
    "Many people came.",          # correct
])

✗ I read a books yesterday.
     └─ [NUM] 'books' → 'book'   ('a' needs a singular noun, but 'books' is plural)
✗ These book are interesting.
     └─ [NUM] 'book' → 'books'   ('These' needs a plural noun, but 'book' is singular)
✗ Every students must attend.
     └─ [NUM] 'students' → 'student'   ('Every' needs a singular noun, but 'students' is plural)
✗ This apples are fresh.
     └─ [NUM] 'apples' → 'apple'   ('This' needs a singular noun, but 'apples' is plural)


✓ I read a few books.
✓ Many people came.


---
### Rule 7 — Commonly confused words (`CONF`)

Some mistakes are *real words used in the wrong place* — a spell-checker cannot see them, but **context** can. We encode a handful of high-frequency patterns:

| Pattern | Fix | Reasoning |
|---|---|---|
| modal + **of** (*could of*) | *could **have*** | "of" is a mis-hearing of *'ve* |
| **their** + *is/are/was/were* | ***there*** | *their* is possessive; needs a noun after it |
| **its** + *been/a/an/not/going* | ***it's*** | *its* = possessive; *it's* = *it is/has* |
| **your** + *going/doing/being/welcome/not/a/the* | ***you're*** | *your* = possessive |
| comparative (*better, more*) + **then** | ***than*** | *than* compares, *then* is time |

In [13]:
MODALS = {"could", "would", "should", "must", "might", "may"}

def check_confusions(doc):
    errors = []
    for tok in doc:
        low = tok.lower_
        nxt = doc[tok.i + 1] if tok.i + 1 < len(doc) else None
        prev = doc[tok.i - 1] if tok.i > 0 else None

        if low in MODALS and nxt is not None and nxt.lower_ == "of":
            errors.append(make_error("CONF", f"'{tok.text} of' is incorrect; it should be '{tok.text} have'",
                                     nxt, suggestion="have"))
        elif low == "their" and nxt is not None and nxt.lower_ in {"is", "are", "was", "were"}:
            errors.append(make_error("CONF", "'their' is possessive; here 'there' is meant",
                                     tok, suggestion=match_case(tok.text, "there")))
        elif low == "its" and nxt is not None and nxt.lower_ in {"been", "a", "an", "not", "going"}:
            errors.append(make_error("CONF", "'its' is possessive; here 'it's' (it is / it has) is meant",
                                     tok, suggestion=match_case(tok.text, "it's")))
        elif low == "your" and nxt is not None and nxt.lower_ in {"going", "doing", "being", "welcome",
                                                                    "not", "a", "an", "the"}:
            errors.append(make_error("CONF", "'your' is possessive; here 'you're' (you are) is meant",
                                     tok, suggestion=match_case(tok.text, "you're")))
        elif low == "then" and prev is not None and prev.tag_ in ("JJR", "RBR"):
            errors.append(make_error("CONF", f"After the comparative '{prev.text}' use 'than', not 'then'",
                                     tok, suggestion=match_case(tok.text, "than")))
    return errors


demo(check_confusions, [
    "She could of won the race.",
    "Their is a problem with the code.",
    "Its been a long day.",
    "Your going to love this.",
    "He is taller then me.",
    "It's been a long day.",       # correct
    "Your car is parked outside.", # correct
])

✗ She could of won the race.
     └─ [CONF] 'of' → 'have'   ('could of' is incorrect; it should be 'could have')
✗ Their is a problem with the code.
     └─ [CONF] 'Their' → 'There'   ('their' is possessive; here 'there' is meant)
✗ Its been a long day.
     └─ [CONF] 'Its' → "It's"   ('its' is possessive; here 'it's' (it is / it has) is meant)
✗ Your going to love this.
     └─ [CONF] 'Your' → "You're"   ('your' is possessive; here 'you're' (you are) is meant)
✗ He is taller then me.
     └─ [CONF] 'then' → 'than'   (After the comparative 'taller' use 'than', not 'then')


✓ It's been a long day.
✓ Your car is parked outside.


---
### Rule 8 — Non-standard word forms (`LEX`)

A **lexicon (dictionary) lookup** for errors that learners make again and again:
* **over-regularised irregular verbs:** *goed → went*, *eated → ate*
* **wrongly pluralised uncountable / irregular nouns:** *informations → information*, *childs → children*
* **missing apostrophes:** *dont → don't*, *cant → can't* (spaCy splits these into `do`+`nt`, so we look for an `nt` token glued to the previous word)
* **lower-case pronoun "i"** → *I*

This is the simplest rule type: no parsing needed, just a dictionary.

In [14]:
NONSTANDARD_WORDS = {
    # over-regularised past tense
    "goed": "went", "runned": "ran", "eated": "ate", "buyed": "bought", "taked": "took",
    "maked": "made", "thinked": "thought", "comed": "came", "bringed": "brought",
    "writed": "wrote", "teached": "taught", "gived": "gave", "knowed": "knew",
    "getted": "got", "drinked": "drank", "speaked": "spoke", "catched": "caught",
    "choosed": "chose", "breaked": "broke", "builded": "built", "swimmed": "swam",
    # wrong plurals
    "childs": "children", "mans": "men", "womans": "women", "foots": "feet",
    "informations": "information", "advices": "advice", "furnitures": "furniture",
    "knowledges": "knowledge", "equipments": "equipment",
    # missing apostrophes (negations like "dont" are handled separately below)
    "im": "I'm", "ive": "I've", "youre": "you're", "alot": "a lot",
}

IRREGULAR_NEGATIONS = {"ca": "can't", "wo": "won't"}     # spaCy splits "cant" → "ca" + "nt"


def check_lexicon(doc):
    errors = []
    for tok in doc:
        # spaCy splits "dont" into "do" + "nt": adjacent tokens with no apostrophe → missing apostrophe
        if tok.lower_ == "nt" and tok.i > 0 and doc[tok.i - 1].idx + len(doc[tok.i - 1].text) == tok.idx:
            prev = doc[tok.i - 1]
            fix = IRREGULAR_NEGATIONS.get(prev.lower_, prev.lower_ + "n't")
            errors.append(make_error("LEX", f"Missing apostrophe: write '{fix}'",
                                     prev, tok, suggestion=match_case(prev.text, fix)))
        elif tok.text == "i":                                   # lower-case pronoun
            errors.append(make_error("LEX", "The pronoun 'I' is always capitalised", tok, suggestion="I"))
        elif tok.lower_ in NONSTANDARD_WORDS:
            fix = NONSTANDARD_WORDS[tok.lower_]
            errors.append(make_error("LEX", f"'{tok.text}' is not standard English; use '{fix}'",
                                     tok, suggestion=match_case(tok.text, fix)))
    return errors


demo(check_lexicon, ["He goed to the market.", "She gave me many informations.",
                     "I dont know the answer.", "She cant swim.", "i like football.",
                     "I don't know."])                      # last one is correct

✗ He goed to the market.
     └─ [LEX] 'goed' → 'went'   ('goed' is not standard English; use 'went')
✗ She gave me many informations.
     └─ [LEX] 'informations' → 'information'   ('informations' is not standard English; use 'information')


✗ I dont know the answer.
     └─ [LEX] 'dont' → "don't"   (Missing apostrophe: write 'don't')
✗ She cant swim.
     └─ [LEX] 'cant' → "can't"   (Missing apostrophe: write 'can't')
✗ i like football.
     └─ [LEX] 'i' → 'I'   (The pronoun 'I' is always capitalised)


✓ I don't know.


---
### Rules 9 & 10 — Capitalisation (`CAP`) and end punctuation (`PUN`)

* **CAP:** a sentence must start with a capital letter. spaCy's parser splits the text into sentences (`doc.sents`), we check the first token of each.
* **PUN:** the last sentence must end with `. ! ?` (or a closing quote/bracket).

*(Lower-case "i" is handled by Rule 8, so we skip it here to avoid reporting the same mistake twice.)*

In [15]:
def check_capitalization(doc):
    errors = []
    for sent in doc.sents:
        first = next((t for t in sent if not t.is_space), None)
        if first is not None and first.is_alpha and first.text.islower() and first.text != "i":
            errors.append(make_error("CAP", "A sentence should start with a capital letter",
                                     first, suggestion=first.text.capitalize()))
    return errors


def check_end_punctuation(doc):
    tokens = [t for t in doc if not t.is_space]
    if len(tokens) < 3:                                  # too short to be sure it is a sentence
        return []
    last = tokens[-1]
    if last.text in {".", "!", "?", "…", '"', "'", ")", "”", "’"}:
        return []
    return [make_error("PUN", "The sentence is missing end punctuation (. ! ?)",
                       last, suggestion=last.text + ".")]


demo(check_capitalization, ["the weather is nice today.", "The weather is nice today."])
demo(check_end_punctuation, ["The weather is nice today", "The weather is nice today."])

✗ the weather is nice today.
     └─ [CAP] 'the' → 'The'   (A sentence should start with a capital letter)


✓ The weather is nice today.
✗ The weather is nice today
     └─ [PUN] 'today' → 'today.'   (The sentence is missing end punctuation (. ! ?))
✓ The weather is nice today.


## 5. Putting it together — the `GrammarChecker`

The checker runs the spaCy pipeline **once**, hands the resulting `Doc` to every rule, and merges the results. On top of detection it offers three views:

* `report()` — readable text
* `highlight()` — HTML with the errors marked (hover for the explanation)
* `correct()` — applies the suggested fixes automatically

For `correct()` we apply fixes **right-to-left** so earlier character positions don't shift, and we skip any fix that overlaps another one. Because fixing one error can reveal another, we repeat for up to 3 passes.

In [16]:
RULES = {
    "SVA": check_subject_verb,
    "ART": check_articles,
    "REP": check_repeated_words,
    "MOD": check_modal_verb,
    "TO":  check_infinitive,
    "NUM": check_determiner_noun,
    "CONF": check_confusions,
    "LEX": check_lexicon,
    "CAP": check_capitalization,
    "PUN": check_end_punctuation,
}


class GrammarChecker:
    def __init__(self, nlp, rules=RULES):
        self.nlp = nlp
        self.rules = rules

    # 1) DETECT ---------------------------------------------------------
    def check(self, text: str) -> List[GrammarError]:
        doc = self.nlp(text)                       # run spaCy once
        errors = []
        for rule_fn in self.rules.values():
            errors.extend(rule_fn(doc))
        errors.sort(key=lambda e: (e.start, e.end))
        return errors

    # 2) REPORT ---------------------------------------------------------
    def report(self, text: str):
        errors = self.check(text)
        print(f"📄 {text}")
        if not errors:
            print("   ✅ No grammar errors found.\n")
            return
        for e in errors:
            fix = f" → '{e.suggestion}'" if e.suggestion is not None else ""
            print(f"   ❌ [{e.rule}] '{e.original}'{fix}\n        {e.message}")
        print()

    # 3) HIGHLIGHT ------------------------------------------------------
    def highlight(self, text: str):
        errors, out, pos = self.check(text), [], 0
        for e in errors:
            if e.start < pos:                      # overlapping → skip
                continue
            out.append(html.escape(text[pos:e.start]))
            tip = html.escape(f"[{e.rule}] {e.message}", quote=True)
            out.append(f"<mark style='background:#ffd6d6;border-bottom:2px solid #d33;padding:0 2px' "
                       f"title=\"{tip}\">{html.escape(text[e.start:e.end] or '␣')}</mark>")
            pos = e.end
        out.append(html.escape(text[pos:]))
        display(HTML("<div style='font-size:16px;line-height:1.7'>" + "".join(out) + "</div>"))

    # 4) CORRECT --------------------------------------------------------
    def correct(self, text: str, max_passes: int = 3) -> str:
        # Several passes: fixing one error can expose another (e.g. "dont" → "don't" → "does"→ …)
        for _ in range(max_passes):
            fixed, limit = text, len(text) + 1
            for e in sorted(self.check(text), key=lambda e: e.start, reverse=True):
                if e.suggestion is None or e.end > limit:       # no fix, or overlaps a later fix
                    continue
                fixed = fixed[:e.start] + e.suggestion + fixed[e.end:]
                limit = e.start
            if fixed == text:
                break
            text = fixed
        return text


checker = GrammarChecker(nlp)
print("Checker ready with rules:", list(RULES))

Checker ready with rules:

 ['SVA', 'ART', 'REP', 'MOD', 'TO', 'NUM', 'CONF', 'LEX', 'CAP', 'PUN']


## 6. Try it yourself

### 6.1 Readable report

In [17]:
samples = [
    "She dont likes apples",
    "He go to school every day.",
    "I saw a elephant at the zoo and it was the the biggest animal.",
    "Their is many books on the shelf, and she could of read them.",
    "The students are preparing for the exam.",
]
for s in samples:
    checker.report(s)

📄 She dont likes apples
   ❌ [SVA] 'do' → 'does'
        Subject 'She' is singular, but verb 'do' does not agree with it
   ❌ [LEX] 'dont' → 'don't'
        Missing apostrophe: write 'don't'
   ❌ [PUN] 'apples' → 'apples.'
        The sentence is missing end punctuation (. ! ?)

📄 He go to school every day.
   ❌ [SVA] 'go' → 'goes'
        Subject 'He' is singular, but verb 'go' does not agree with it



📄 I saw a elephant at the zoo and it was the the biggest animal.
   ❌ [ART] 'a' → 'an'
        Use 'an' before 'elephant' (it starts with a vowel sound)
   ❌ [REP] 'the the' → 'the'
        The word 'the' is repeated

📄 Their is many books on the shelf, and she could of read them.
   ❌ [CONF] 'Their' → 'There'
        'their' is possessive; here 'there' is meant
   ❌ [CONF] 'of' → 'have'
        'could of' is incorrect; it should be 'could have'

📄 The students are preparing for the exam.
   ✅ No grammar errors found.



### 6.2 Highlighted text *(hover over a red word to see the explanation)*

In [18]:
checker.highlight("she dont like apples. He go to a university every day and I want to went home")

### 6.3 Automatic correction

In [19]:
for s in samples:
    print("BEFORE:", s)
    print("AFTER :", checker.correct(s))
    print()

BEFORE: She dont likes apples
AFTER : She doesn't likes apples.

BEFORE: He go to school every day.
AFTER : He goes to school every day.

BEFORE: I saw a elephant at the zoo and it was the the biggest animal.
AFTER : I saw an elephant at the zoo and it was the biggest animal.

BEFORE: Their is many books on the shelf, and she could of read them.
AFTER : There are many books on the shelf, and she could have read them.

BEFORE: The students are preparing for the exam.
AFTER : The students are preparing for the exam.



### 6.4 Interactive mode
Type your own sentence (leave blank to stop).

In [ ]:
while True:
    text = input("Enter a sentence (blank to quit): ").strip()
    if not text:
        break
    checker.report(text)
    print("   ✏️  Suggested:", checker.correct(text), "\n")

## 7. Evaluation

A detector is only useful if we can **measure** it. We build **two** small hand-labelled sets; each sentence has the **set of rule codes** that *should* fire. Sentences with `set()` are grammatically correct — they test for **false alarms**.

* **Development set** — sentences we looked at *while writing and debugging* the rules. Scores here are optimistic (we fixed whatever failed).
* **Held-out set** — new sentences written *afterwards* and **never used to tune anything**. This is the honest estimate of how the checker behaves on unseen text.

We evaluate **per rule** at sentence level:

* **TP** — rule expected and fired
* **FP** — rule fired but was *not* expected (false alarm)
* **FN** — rule expected but did *not* fire (missed error)

$$\text{Precision}=\frac{TP}{TP+FP}\qquad \text{Recall}=\frac{TP}{TP+FN}\qquad F_1=\frac{2PR}{P+R}$$

*Precision* = "when it flags something, is it right?" · *Recall* = "does it catch the errors that exist?"

### 7.1 Development set

In [21]:
# DEVELOPMENT SET — (sentence, set of rule codes that should fire)
DEV_SET = [
    # ---- Subject–verb agreement
    ("He go to school every day.",                          {"SVA"}),
    ("The students was late for class.",                    {"SVA"}),
    ("She don't like coffee.",                              {"SVA"}),
    ("They goes to the market.",                            {"SVA"}),
    ("I is ready for the exam.",                            {"SVA"}),
    ("There is many books on the shelf.",                   {"SVA"}),
    ("My brother and sister likes pizza.",                  {"SVA"}),
    # ---- Articles
    ("She bought a apple yesterday.",                       {"ART"}),
    ("He is an university student.",                        {"ART"}),
    ("I saw a elephant at the zoo.",                        {"ART"}),
    # ---- Repeated words
    ("I went to to the market.",                            {"REP"}),
    ("This is the the best project.",                       {"REP"}),
    # ---- Modal + verb form
    ("She can goes to the party.",                          {"MOD"}),
    ("He will went home tomorrow.",                         {"MOD"}),
    ("We must finished the work today.",                    {"MOD"}),
    # ---- Infinitive
    ("I want to went home.",                                {"TO"}),
    ("She decided to goes there.",                          {"TO"}),
    # ---- Determiner–noun number
    ("I read a books yesterday.",                           {"NUM"}),
    ("These book are interesting.",                         {"NUM"}),
    ("Every students must attend.",                         {"NUM"}),
    ("This apples are fresh.",                              {"NUM"}),
    # ---- Confusions
    ("She could of won the race.",                          {"CONF"}),
    ("Their is a problem with the code.",                   {"CONF"}),
    ("Its been a long day.",                                {"CONF"}),
    ("Your going to love this.",                            {"CONF"}),
    ("He is taller then me.",                               {"CONF"}),
    # ---- Lexicon
    ("He goed to the market.",                              {"LEX"}),
    ("She gave me many informations.",                      {"LEX"}),
    ("I dont know the answer.",                             {"LEX"}),
    ("i like football.",                                    {"LEX"}),
    # ---- Capitalisation / punctuation
    ("the weather is nice today.",                          {"CAP"}),
    ("The weather is nice today",                           {"PUN"}),
    # ---- Multiple errors in one sentence
    ("she dont like apples",                                {"CAP", "LEX", "SVA", "PUN"}),
    ("The boys plays a game and he go home.",               {"SVA"}),
    # ---- Correct sentences (must NOT fire)
    ("The cat sleeps on the sofa.",                         set()),
    ("They are playing football in the park.",              set()),
    ("She has finished her homework.",                      set()),
    ("If I were you, I would study harder.",                set()),
    ("A number of students are absent today.",              set()),
    ("The books that he reads are interesting.",            set()),
    ("He doesn't like spicy food.",                         set()),
    ("We have an honest teacher.",                          set()),
    ("Everyone is ready.",                                  set()),
    ("Tom and Jerry are friends.",                          set()),
    ("She can speak three languages.",                      set()),
    ("I want to learn natural language processing.",        set()),
    ("Your car is parked outside.",                         set()),
    ("It's been a long day.",                               set()),
    ("A university is a place of learning.",                set()),
    ("There are many books on the shelf.",                  set()),
    ("The data shows a clear trend.",                       set()),
    ("She will be going to Chennai.",                       set()),
    ("We should have gone earlier.",                        set()),
    ("The team has won every match this season.",           set()),
    ("Where is the library?",                               set()),
    ("Do they know the answer?",                            set()),
    ("He said that that was fine.",                         set()),
    ("I read a few books last month.",                      set()),
    ("She is an engineer and he is a doctor.",              set()),
]
print("Development set:", len(DEV_SET),
      "| with errors:", sum(1 for _, e in DEV_SET if e),
      "| correct:", sum(1 for _, e in DEV_SET if not e))

Development set:

 59 | with errors: 34 | correct: 25


In [22]:
def evaluate(checker, test_set):
    stats = {r: {"TP": 0, "FP": 0, "FN": 0} for r in RULES}
    details = []                                           # for error analysis
    for sentence, expected in test_set:
        predicted = {e.rule for e in checker.check(sentence)}
        for r in RULES:
            if r in expected and r in predicted:     stats[r]["TP"] += 1
            elif r in predicted:                     stats[r]["FP"] += 1; details.append(("FALSE ALARM", r, sentence))
            elif r in expected:                      stats[r]["FN"] += 1; details.append(("MISSED",      r, sentence))
    rows = []
    for r, s in stats.items():
        tp, fp, fn = s["TP"], s["FP"], s["FN"]
        p = tp / (tp + fp) if tp + fp else float("nan")
        rc = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2 * p * rc / (p + rc) if tp else (0.0 if tp + fp + fn else float("nan"))
        rows.append({"Rule": r, "TP": tp, "FP": fp, "FN": fn,
                     "Precision": round(p, 2), "Recall": round(rc, 2), "F1": round(f1, 2)})
    df = pd.DataFrame(rows).set_index("Rule")
    # micro-average over all rules
    TP, FP, FN = df["TP"].sum(), df["FP"].sum(), df["FN"].sum()
    P, R = TP / (TP + FP), TP / (TP + FN)
    micro = {"TP": TP, "FP": FP, "FN": FN, "Precision": round(P, 2),
             "Recall": round(R, 2), "F1": round(2 * P * R / (P + R), 2)}
    df.loc["OVERALL (micro)"] = micro
    return df, details


dev_results, dev_details = evaluate(checker, DEV_SET)
print("DEVELOPMENT SET (optimistic — used while building the rules)")
dev_results

DEVELOPMENT SET (optimistic — used while building the rules)


,TP,FP,FN,Precision,Recall,F1
Rule,,,,,,
SVA,9,0,0,1.0,1.0,1.0
ART,3,0,0,1.0,1.0,1.0
REP,2,0,0,1.0,1.0,1.0
MOD,3,0,0,1.0,1.0,1.0
TO,2,0,0,1.0,1.0,1.0
NUM,4,0,0,1.0,1.0,1.0
CONF,5,0,0,1.0,1.0,1.0
LEX,5,0,0,1.0,1.0,1.0
CAP,2,0,0,1.0,1.0,1.0


### 7.2 Held-out set — the honest test
New sentences, written after the rules were finished.

In [23]:
# HELD-OUT SET — written AFTER the rules were finished; never used for tuning
HELD_OUT = [
    # ---- errors
    ("The children plays in the garden.",               {"SVA"}),
    ("My friends is coming to the party.",              {"SVA"}),
    ("He have a new bike.",                             {"SVA"}),
    ("We was happy with the result.",                   {"SVA"}),
    ("She bought an new laptop.",                       {"ART"}),
    ("It was a honest mistake.",                        {"ART"}),
    ("He ate a orange.",                                {"ART"}),
    ("The the teacher explained the lesson.",           {"REP"}),
    ("They might arrived late.",                        {"MOD"}),
    ("I should studied more.",                          {"MOD"}),
    ("He wants to eats lunch.",                         {"TO"}),
    ("We plan to visited Kerala.",                      {"TO"}),
    ("She has many friend in Delhi.",                   {"NUM"}),
    ("I need a pens.",                                  {"NUM"}),
    ("Those car are very old.",                         {"NUM"}),
    ("He should of told me.",                           {"CONF"}),
    ("This is better then that.",                       {"CONF"}),
    ("The dog wagged it's tail.",                       {"CONF"}),
    ("He eated all the rice.",                          {"LEX"}),
    ("They gived me two advices.",                      {"LEX"}),
    ("I cant believe it.",                              {"LEX"}),
    ("yesterday was a good day.",                       {"CAP"}),
    ("What a lovely day",                               {"PUN"}),
    ("we dont have time",                               {"CAP", "LEX", "PUN"}),
    # ---- correct
    ("My sister works at a hospital.",                  set()),
    ("The children are playing outside.",               set()),
    ("She doesn't know the answer.",                    set()),
    ("Neither of them was late.",                       set()),
    ("The news is good.",                               set()),
    ("Many students have already left.",                set()),
    ("He can swim and she can dance.",                  set()),
    ("We are going to the market tomorrow.",            set()),
    ("Machine learning models need a lot of data.",     set()),
    ("He is an honorable man.",                         set()),
    ("She wants to be a teacher.",                      set()),
    ("Every student has a laptop.",                     set()),
    ("Those books belong to me.",                       set()),
    ("This is better than that.",                       set()),
    ("I think that it's time to go.",                   set()),
    ("The results were surprising.",                    set()),
    ("Do you have an umbrella?",                        set()),
    ("My parents live in Kochi.",                       set()),
    ("Python is a popular language.",                   set()),
    ("A European tour is planned.",                     set()),
    ("Who is coming to dinner?",                        set()),
    ("These are my friends.",                           set()),
    ("The old man and his dog walk every morning.",     set()),
    ("Time flies when you are having fun.",             set()),
]

print("Held-out set:", len(HELD_OUT))

Held-out set: 48


In [24]:
held_results, held_details = evaluate(checker, HELD_OUT)
print("HELD-OUT SET (unseen sentences)")
held_results

HELD-OUT SET (unseen sentences)


,TP,FP,FN,Precision,Recall,F1
Rule,,,,,,
SVA,4,0,0,1.0,1.00,1.00
ART,3,0,0,1.0,1.00,1.00
REP,1,0,0,1.0,1.00,1.00
MOD,2,0,0,1.0,1.00,1.00
TO,2,0,0,1.0,1.00,1.00
NUM,3,0,0,1.0,1.00,1.00
CONF,2,0,1,1.0,0.67,0.80
LEX,4,0,0,1.0,1.00,1.00
CAP,2,0,0,1.0,1.00,1.00


### 7.3 Extra metrics (both sets combined)

In [25]:
# Bonus metric: how often is a completely CORRECT sentence left alone?
ALL_SETS = DEV_SET + HELD_OUT
correct_sents = [s for s, e in ALL_SETS if not e]
clean = sum(1 for s in correct_sents if not checker.check(s))
print(f"Correct sentences left untouched: {clean}/{len(correct_sents)} ({clean/len(correct_sents):.0%})")

# Bonus metric: does auto-correction really fix the sentence? (re-check the corrected text)
wrong_sents = [s for s, e in ALL_SETS if e]
fixed_ok = sum(1 for s in wrong_sents if not checker.check(checker.correct(s)))
print(f"Erroneous sentences that are error-free after auto-correct: {fixed_ok}/{len(wrong_sents)} ({fixed_ok/len(wrong_sents):.0%})")

Correct sentences left untouched: 49/49 (100%)


Erroneous sentences that are error-free after auto-correct: 57/58 (98%)


## 8. Error analysis & limitations

The most valuable part of an evaluation is looking at **what went wrong**.

In [26]:
all_details = dev_details + held_details
if not all_details:
    print("🎉 No misses or false alarms!")
for kind, rule, sentence in all_details:
    print(f"{kind:<12} [{rule}]  {sentence}")
    print("             detected:", [(e.rule, e.original) for e in checker.check(sentence)])

MISSED       [CONF]  The dog wagged it's tail.
             detected: []


### Why does it sometimes fail? (limitations of the approach)

| Limitation | Explanation |
|---|---|
| **Depends on spaCy's tagger/parser** | If the parser mis-reads a grammatically *wrong* sentence (it was trained on correct text), our rules inherit that mistake. |
| **Rules cover only known patterns** | Tense consistency, word order, missing words, prepositions (*"depends of"*), etc. are not detected. |
| **No real semantics** | The checker knows *form*, not *meaning* — *"The idea sleeps furiously"* is grammatical but nonsense. |
| **Heuristic for *a/an*** | Based on spelling + a short exception list; rare words can be mis-classified. |
| **Known misses (seen in the evaluation)** | *"The dog wagged **it's** tail"* — the reverse confusion (*it's* for possessive *its*) is not covered. *"many informations"* is repaired to *"many information"*, but the correct fix is *"much information"* — we don't model countable vs. uncountable nouns. |
| **Small, hand-made test set** | Scores are indicative, not a benchmark. For a serious comparison use CoNLL-2014 / BEA-2019 / JFLEG. |
| **Precision vs. recall trade-off** | We deliberately skip uncertain cases (e.g. collective nouns, *"a number of"*) to avoid annoying false alarms. |

**Design lesson:** in a grammar checker, *false alarms hurt more than misses* — users stop trusting a tool that flags correct text. That is why most rules have **guards** (exception lists and "when unsure, stay silent").

## 9. Future work

1. **More rules** — tense consistency (*"Yesterday I go"*), preposition errors, missing auxiliary (*"She going"*), comparative/superlative forms (*"more better"*).
2. **Spell checking** — add edit-distance (Levenshtein) candidates against a vocabulary.
3. **Statistical layer** — use an n-gram language model (or a masked LM) to score "how surprising is this word here?" and combine it with the rules.
4. **Learn the rules** — train a classifier on an error-annotated corpus (e.g. BEA-2019) using the same spaCy features (`tag_`, `dep_`, `lemma_`).
5. **Better evaluation** — span-level precision/recall (ERRANT-style) on a public benchmark.
6. **Web / Gradio UI** — wrap `GrammarChecker.highlight()` in a small app.

---
## Conclusion
We built a transparent grammar-error detector from scratch on top of spaCy's linguistic features:

* **10 explainable rules** covering agreement, articles, verb forms, number, confusables, lexical errors, capitalisation and punctuation
* a clean architecture: **spaCy → rules → `GrammarError` → report / highlight / correct**
* a measurable evaluation (**precision / recall / F1**) with error analysis

Because every decision is a small, readable rule, the system can always answer **"why did you flag this?"** — something black-box neural models cannot do.

### References
* spaCy documentation — <https://spacy.io/usage/linguistic-features>
* Penn Treebank POS tag set — Marcus et al., 1993
* Bryant et al., *The BEA-2019 Shared Task on Grammatical Error Correction*, 2019
* Ng et al., *The CoNLL-2014 Shared Task on Grammatical Error Correction*, 2014